# Funding Valuation Adjustment (FVA)
## QRE-54

FVA is the present value of **funding costs and benefits** arising from an uncollateralised (or partially collateralised) derivative position. Where CVA prices counterparty default risk, FVA prices the cost of capital that must be tied up to support a trade that lacks a Credit Support Annex (CSA).

$$V^{\text{risky}} = V^{\text{risk-free}} - \text{CVA} - \text{FVA}$$

**FVA decomposes into:**

$$\text{FVA} = \underbrace{\text{FCA}}_{\text{Funding Cost Adj.}} - \underbrace{\text{FBA}}_{\text{Funding Benefit Adj.}}$$

- **FCA** — cost of funding the positive exposure (we need to borrow at spread above OIS when the trade is in our favour and not collateralised)
- **FBA** — benefit of reinvesting the negative exposure (when the trade is against us and we receive or hold uncollateralised cash)

**Infrastructure consumed:** same `MCSimulator` and exposure profile from QRE-53 — FVA replaces the credit default weighting with a funding spread weighting.

**Key design:** all funding parameters (spread, CSA type, direction) are function arguments. Changing the collateral arrangement is one argument swap.


In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from quant_risk.setup import base
from quant_risk.config import PROCESSED_DIR
from quant_risk.curves.ois import OISCurve
from quant_risk.models.rates import HullWhiteProcess
from quant_risk.models.simulator import MCSimulator

np, pd, plt = base()

RNG_SEED = 42


---
## 1. FVA Theory — Piterbarg (2010) Framework

### 1.1 Why Funding Costs Matter

Consider a bank that enters an uncollateralised payer swap. When the swap is in-the-money ($V(t) > 0$), the bank holds an unsecured receivable. To hedge the market risk, the bank needs to:

1. Pay fixed on an **offsetting collateralised swap** with a dealer (CSA in place → funded at OIS)
2. Fund the **collateral shortfall** — the gap between what the CSA swap requires and what the uncollateralised client provides

The bank must borrow this collateral in the repo/interbank market at its **unsecured funding rate** $r_{\text{fund}} = r_{\text{OIS}} + s_f$. The spread $s_f$ is the FVA cost.

### 1.2 Discrete FVA Formula

$$\text{FCA} = s_f \sum_{i=1}^n \Delta t_i \cdot \underbrace{\mathbb{E}^{\mathbb{Q}}[D(0,t_i)\max(V(t_i),0)]}_{\text{EE}_{\text{disc}}(t_i)}$$

$$\text{FBA} = s_r \sum_{i=1}^n \Delta t_i \cdot \underbrace{\mathbb{E}^{\mathbb{Q}}[D(0,t_i)|\min(V(t_i),0)|]}_{|\text{NEE}_{\text{disc}}(t_i)|}$$

$$\text{FVA} = \text{FCA} - \text{FBA}$$

where:
- $s_f$ — funding spread: bank's cost of **unsecured borrowing** above OIS (bps)
- $s_r$ — reinvestment spread: return earned above OIS on uncollateralised negative exposure (bps)
- $\Delta t_i$ — period length in years

### 1.3 How CSA Type Determines FVA

| CSA arrangement | FCA | FBA | FVA |
|---|---|---|---|
| **Two-way CSA** (daily VM both ways) | 0 | 0 | **0** |
| **One-way CSA** (we post, they don't) | $s_f \times \sum \Delta t_i \cdot \text{EE}_{\text{disc}}$ | 0 | = FCA |
| **No CSA** (uncollateralised) | $s_f \times \sum \Delta t_i \cdot \text{EE}_{\text{disc}}$ | $s_r \times \sum \Delta t_i \cdot |\text{NEE}_{\text{disc}}|$ | FCA − FBA |

For a symmetric funding spread ($s_f = s_r = s$), the no-CSA formula simplifies to:

$$\text{FVA}^{\text{no CSA}} = s \sum_{i} \Delta t_i \cdot \mathbb{E}^{\mathbb{Q}}[D(0,t_i)\,V(t_i)]$$

This is just the funding spread times the **discounted expected future MTM** — confirming that FVA is zero for an ATM trade at inception only in expectation, not path-by-path.

### 1.4 OIS vs SOFR Funding

Post-LIBOR transition (EUR: ESTR, USD: SOFR), cash collateral under CSA earns the overnight risk-free rate. The funding spread $s_f$ is now measured relative to ESTR/SOFR:

- EUR bank with $s_f = 30$ bps: funding costs 30 bps/year above ESTR on uncollateralised positions
- EUR/USD cross-currency swap: the EUR-USD basis swap spread enters the funding spread for the non-domestic currency leg
- IBOR-legacy positions: existing LIBOR CSAs being novated to SOFR/ESTR may carry a CAS (Credit Adjustment Spread) — a one-time FVA-like adjustment


In [ ]:
# ── Load OIS, build HW simulator, build exposure profile ─────────────────────
# Replicates the QRE-53 setup: same trade, same process, same profile.
# The CVA and FVA notebooks share this infrastructure — in production they run
# on the same MCSimulator instance for the same netting set.

try:
    ois = OISCurve.from_processed(str(PROCESSED_DIR))
    print(ois.describe())
except FileNotFoundError:
    print("Using synthetic OIS curve")
    data = pd.DataFrame(
        {"years":           [1/12,2/12,3/12,6/12,9/12,1.0,2.0,3.0,5.0,10.0,15.0],
         "zero_rate_pct":   [2.63,2.61,2.58,2.48,2.38,2.30,2.20,2.15,2.20,2.40,2.50],
         "discount_factor": [np.exp(-r/100*t) for r,t in zip(
             [2.63,2.61,2.58,2.48,2.38,2.30,2.20,2.15,2.20,2.40,2.50],
             [1/12,2/12,3/12,6/12,9/12,1,2,3,5,10,15])],
         "valuation_date":  ["2026-03-24"]*11},
        index=["1M","2M","3M","6M","9M","12M","2Y","3Y","5Y","10Y","10Y+"])
    data.index.name = "maturity"
    ois = OISCurve(data)

KAPPA, SIGMA = 0.10, 0.50
r0 = ois.forward_rate(1/12, 2/12)

sim = MCSimulator(
    process   = HullWhiteProcess(curve=ois, kappa=KAPPA, sigma=SIGMA),
    x0        = r0,
    T         = 10.0,
    n_steps   = 120,
    n_paths   = 5000,
    antithetic= True,
    seed      = RNG_SEED,
)
print(sim.describe())
print(f"r(0) = {r0:.4f}%")


In [ ]:
# ── HW bond price and IRS MTM — same functions as QRE-53 (all args) ──────────

def hw_B(tau, kappa):
    return np.where(tau > 0, (1 - np.exp(-kappa * tau)) / kappa, 0.0)

def hw_bond_price_paths(t, maturities, r_t, kappa, sigma, curve):
    tau    = maturities - t
    B_tau  = hw_B(tau, kappa)
    P_0T   = np.array([curve.discount_factor(T) for T in maturities])
    P_0t   = curve.discount_factor(t) if t > 1e-6 else 1.0
    dt_fd  = 1/12; t_lo = max(t - dt_fd, dt_fd)
    f_0t   = curve.forward_rate(t_lo, t_lo + dt_fd)
    sigma_d = sigma / 100
    var_adj = (sigma_d**2 / (4*kappa)) * B_tau**2 * (1 - np.exp(-2*kappa*t))
    rate_dev = (r_t[:, None] - f_0t) / 100 * B_tau[None, :]
    return (P_0T / P_0t)[None, :] * np.exp(-rate_dev - var_adj[None, :])

def irs_mtm(paths, t, payment_dates, year_fracs,
             K, notional, is_payer, kappa, sigma, curve, dt, n_steps):
    remaining = payment_dates[payment_dates > t]
    yf_r      = year_fracs[payment_dates > t]
    if len(remaining) == 0:
        return np.zeros(paths.shape[0])
    t_idx  = min(int(round(t / dt)), n_steps)
    r_t    = paths[:, t_idx]
    P_ti   = hw_bond_price_paths(t, remaining, r_t, kappa, sigma, curve)
    float_leg = 1.0 - P_ti[:, -1]
    fixed_leg = (K / 100) * (yf_r * P_ti).sum(axis=1)
    payer_mtm = notional * (float_leg - fixed_leg)
    return payer_mtm if is_payer else -payer_mtm


# ── Trade parameters ──────────────────────────────────────────────────────────
SWAP_MATURITY   = 5.0
SWAP_COUPON     = r0
SWAP_NOTIONAL   = 1_000_000
IS_PAYER        = True

payment_dates = np.arange(1.0, SWAP_MATURITY + 0.001, 1.0)
year_fracs    = np.ones(len(payment_dates))
exp_dates     = np.arange(0.5, SWAP_MATURITY + 0.5, 0.5)

def mtm_fn(paths, t):
    return irs_mtm(paths, t, payment_dates, year_fracs,
                   SWAP_COUPON, SWAP_NOTIONAL, IS_PAYER,
                   KAPPA, SIGMA, ois, sim.dt, sim.n_steps)

profile = sim.exposure_profile(mtm_fn, exp_dates)

# Pre-compute discounted NEE from the raw MTM matrix (needed for FBA)
disc_matrix = np.column_stack([sim.sdf(t) for t in exp_dates])   # (n_paths, n_dates)
nee_disc    = (disc_matrix * np.minimum(profile['mtm'], 0)).mean(axis=0)

print(f"Exposure profile ready:")
print(f"  EPE (time-avg EE)    = {profile['EPE']:>10,.2f} EUR")
print(f"  Peak EE_disc         = {profile['EE_disc'].max():>10,.2f} EUR  at t={exp_dates[profile['EE_disc'].argmax()]}Y")
print(f"  Peak |NEE_disc|      = {np.abs(nee_disc).max():>10,.2f} EUR  at t={exp_dates[np.abs(nee_disc).argmax()]}Y")


In [ ]:
# ── FVA calculation — all parameters as arguments ────────────────────────────

def compute_fva(
    ee_disc: np.ndarray,
    nee_disc: np.ndarray,
    exp_dates: np.ndarray,
    funding_spread_bps: float,
    invest_spread_bps: float,
    csa_type: str,
) -> dict:
    """
    FCA = s_f × Σ Δtᵢ × EE_disc(tᵢ)
    FBA = s_r × Σ Δtᵢ × |NEE_disc(tᵢ)|
    FVA = FCA - FBA

    Parameters
    ----------
    ee_disc            : shape (n_dates,) — discounted EE from MCSimulator
    nee_disc           : shape (n_dates,) — discounted NEE (E[D(0,t) min(V,0)])
    exp_dates          : evaluation dates in years
    funding_spread_bps : bank's cost of unsecured funding above OIS (bps/year)
    invest_spread_bps  : return earned on uncollateralised cash above OIS (bps/year)
    csa_type           : 'none'            — no CSA (uncollateralised)
                         'one_way_post'    — we always post; they never post
                         'one_way_receive' — they always post; we never post
                         'two_way'         — full two-way daily variation margin

    Returns
    -------
    dict with FVA, FCA, FBA and per-period arrays.
    """
    s_f = funding_spread_bps / 10_000    # decimal/year
    s_r = invest_spread_bps  / 10_000

    t_prev = np.concatenate([[0.0], exp_dates[:-1]])
    dt_i   = exp_dates - t_prev          # period lengths in years

    # EE_disc and |NEE_disc| are the funding exposure components
    abs_nee_disc = np.abs(nee_disc)

    if csa_type == 'two_way':
        # Fully collateralised: no net funding requirement
        fca_by_period = np.zeros_like(ee_disc)
        fba_by_period = np.zeros_like(ee_disc)

    elif csa_type == 'one_way_post':
        # We post when V < 0; they don't post when V > 0.
        # When V > 0: we need to fund the receivable → FCA
        # When V < 0: we post collateral at OIS (no spread cost) → no FBA
        fca_by_period = s_f * dt_i * ee_disc
        fba_by_period = np.zeros_like(ee_disc)

    elif csa_type == 'one_way_receive':
        # They post when V < 0 (from their side); we don't post.
        # When V > 0: we fund the receivable → FCA
        # When V < 0: they post cash we can reinvest at OIS + s_r → FBA
        fca_by_period = s_f * dt_i * ee_disc
        fba_by_period = s_r * dt_i * abs_nee_disc

    else:  # 'none' — no CSA
        fca_by_period = s_f * dt_i * ee_disc
        fba_by_period = s_r * dt_i * abs_nee_disc

    fca = float(fca_by_period.sum())
    fba = float(fba_by_period.sum())
    fva = fca - fba

    return {
        'FVA'           : fva,
        'FCA'           : fca,
        'FBA'           : fba,
        'fca_by_period' : fca_by_period,
        'fba_by_period' : fba_by_period,
    }


# ── Reference parameters ──────────────────────────────────────────────────────
FUNDING_SPREAD_BPS = 30.0   # EUR A-rated bank: 30 bps above ESTR
INVEST_SPREAD_BPS  = 20.0   # slightly lower reinvestment spread

result_no_csa = compute_fva(
    profile['EE_disc'], nee_disc, exp_dates,
    FUNDING_SPREAD_BPS, INVEST_SPREAD_BPS, 'none'
)

print(f"FVA Summary — no CSA, s_f={FUNDING_SPREAD_BPS:.0f} bps, s_r={INVEST_SPREAD_BPS:.0f} bps")
print(f"  FCA = {result_no_csa['FCA']:>10,.2f} EUR  ({result_no_csa['FCA']/SWAP_NOTIONAL*10000:.2f} bps)")
print(f"  FBA = {result_no_csa['FBA']:>10,.2f} EUR  ({result_no_csa['FBA']/SWAP_NOTIONAL*10000:.2f} bps)")
print(f"  FVA = {result_no_csa['FVA']:>10,.2f} EUR  ({result_no_csa['FVA']/SWAP_NOTIONAL*10000:.2f} bps)")


In [ ]:
# ── FVA profile: FCA and FBA contributions by period ─────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: EE_disc vs |NEE_disc| — the inputs to FCA and FBA
ax = axes[0]
ax.fill_between(exp_dates, 0, profile['EE_disc'] / 1e4,
                alpha=0.30, color='firebrick',  label='EE_disc → FCA')
ax.fill_between(exp_dates, 0, np.abs(nee_disc) / 1e4,
                alpha=0.30, color='steelblue', label='|NEE_disc| → FBA')
ax.plot(exp_dates, profile['EE_disc'] / 1e4, '-',  color='firebrick',  lw=2.0)
ax.plot(exp_dates, np.abs(nee_disc)   / 1e4, '--', color='steelblue', lw=2.0)
direction = 'Payer' if IS_PAYER else 'Receiver'
ax.set_xlabel('t (years)')
ax.set_ylabel('Discounted exposure (EUR × 10⁴)')
ax.set_title(f'{direction} IRS — discounted EE and |NEE|\n'
             f'(EE_disc → FCA cost;  |NEE_disc| → FBA benefit)')
ax.legend(fontsize=7)

# Right: FCA and FBA by period — waterfall
ax = axes[1]
w = 0.18
ax.bar(exp_dates - w, result_no_csa['fca_by_period'],
       width=w*2, color='firebrick', alpha=0.8, label='FCA contribution')
ax.bar(exp_dates + w, result_no_csa['fba_by_period'],
       width=w*2, color='steelblue', alpha=0.8, label='FBA contribution')
fva_cumsum = (result_no_csa['fca_by_period'] - result_no_csa['fba_by_period']).cumsum()
ax.plot(exp_dates, fva_cumsum, 'k--', lw=1.5, marker='o', ms=4,
        label=f'Cumulative FVA → {result_no_csa["FVA"]:,.0f} EUR')
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('Period end t (years)')
ax.set_ylabel('Adjustment (EUR)')
ax.set_title(f'FVA contributions: FCA − FBA = {result_no_csa["FVA"]:,.2f} EUR '
             f'({result_no_csa["FVA"]/SWAP_NOTIONAL*10000:.1f} bps)\n'
             f's_f={FUNDING_SPREAD_BPS:.0f} bps, s_r={INVEST_SPREAD_BPS:.0f} bps, no CSA')
ax.legend(fontsize=7)

plt.tight_layout()
plt.show()


---
## 2. CSA Impact — Quantifying the Value of Collateral

### 2.1 What a CSA Does

A **Credit Support Annex** (CSA) is a bilateral agreement to post variation margin daily. Under a two-way CSA with zero threshold:
- The in-the-money party always receives cash collateral equal to the current MTM
- The out-of-the-money party posts at OIS (for cash CSA) — no funding spread

This **eliminates FVA** entirely. The value of entering a CSA (relative to no-CSA) is exactly the FVA of the uncollateralised trade.

### 2.2 CSA Threshold and Independent Amount

Real CSAs have:
- **Threshold** $H$: collateral is only called when $|V| > H$. For $H > 0$, partial FVA remains.
- **Minimum Transfer Amount** (MTA): practical minimum to avoid operationally costly micro-transfers
- **Independent Amount** (IM): upfront collateral (equivalent to initial margin in SIMM)

A threshold of $H$ effectively means the uncollateralised portion is $\min(V(t), H)$ for positive exposure. For the notebook we model the two polar cases (zero threshold and infinite threshold).

### 2.3 Regulatory Margin Rules (EMIR)

Under EMIR Refit and Basel III margin rules:
- **Uncleared OTC derivatives**: bilateral IM (independent amount) required since phase-in dates
- **Variation Margin**: mandatory for financial counterparties above the clearing threshold
- **CSA on cleared trades**: LCH/Eurex clearing provides the equivalent of a perfect two-way CSA

For trades subject to mandatory clearing (e.g., vanilla EUR IRS > EUR 50M notional): FVA ≈ 0 because variation margin is posted daily at OIS.


In [ ]:
# ── FVA under different CSA arrangements ──────────────────────────────────────

csa_cases = [
    ('two_way',        'Two-way CSA (mandatory clearing)'),
    ('one_way_receive','One-way CSA (they post, we don't)'),
    ('one_way_post',   'One-way CSA (we post, they don't)'),
    ('none',           'No CSA (uncollateralised)'),
]

print(f"FVA by CSA type — s_f={FUNDING_SPREAD_BPS:.0f} bps, s_r={INVEST_SPREAD_BPS:.0f} bps")
print(f"{'CSA type':<40}  {'FCA':>10}  {'FBA':>10}  {'FVA':>10}  {'(bps)':>8}")
print("─" * 80)

csa_results = {}
for csa_type, label in csa_cases:
    res = compute_fva(profile['EE_disc'], nee_disc, exp_dates,
                      FUNDING_SPREAD_BPS, INVEST_SPREAD_BPS, csa_type)
    csa_results[csa_type] = res
    bps = res['FVA'] / SWAP_NOTIONAL * 10000
    print(f"  {label:<38}  {res['FCA']:>10,.2f}  {res['FBA']:>10,.2f}  "
          f"{res['FVA']:>10,.2f}  {bps:>8.2f}")

print("─" * 80)
print(f"  Value of two-way CSA = FVA(none) − FVA(two-way) = "
      f"{(csa_results['none']['FVA'] - csa_results['two_way']['FVA']):,.2f} EUR")

# ── Plot: FVA by CSA type ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
labels  = [lbl for _, lbl in csa_cases]
fva_vals = [csa_results[ct]['FVA'] / SWAP_NOTIONAL * 10000 for ct, _ in csa_cases]
fca_vals = [csa_results[ct]['FCA'] / SWAP_NOTIONAL * 10000 for ct, _ in csa_cases]
fba_vals = [csa_results[ct]['FBA'] / SWAP_NOTIONAL * 10000 for ct, _ in csa_cases]

x = np.arange(len(labels))
w = 0.25
ax.bar(x - w,   fca_vals, width=w*2, color='firebrick', alpha=0.8, label='FCA')
ax.bar(x + w,  [-f for f in fba_vals], width=w*2, color='steelblue', alpha=0.8,
       label='FBA (benefit, shown negative)')
ax.plot(x, fva_vals, 'ko', ms=8, zorder=5, label='Net FVA')
for xi, fv in zip(x, fva_vals):
    ax.annotate(f'{fv:.1f}', (xi, fv), textcoords='offset points',
                xytext=(0, 8), ha='center', fontsize=8, fontweight='bold')
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=12, ha='right', fontsize=8)
ax.set_ylabel('FVA (bps of notional)')
ax.set_title(f'FVA by CSA type — {direction} IRS, K={SWAP_COUPON:.2f}%, '
             f'{SWAP_MATURITY:.0f}Y, N={SWAP_NOTIONAL/1e6:.0f}M EUR')
ax.legend(fontsize=7)
plt.tight_layout()
plt.show()


---
## 3. Funding Spread Sensitivity and OIS vs SOFR Context

### 3.1 What the Funding Spread Represents

The funding spread $s_f$ above OIS reflects the **bank's credit quality** in the unsecured interbank market. Post-LIBOR transition:

| Rate | Use | Typical 1Y spread above OIS (2026) |
|---|---|---|
| ESTR (€STR) | EUR OIS discounting, EUR CSA collateral | Benchmark — 0 bps by definition |
| EURIBOR 3M | Legacy floating coupons | 7–15 bps ESTR-EURIBOR basis |
| EUR senior unsecured (AA bank) | FVA funding rate | 15–30 bps above ESTR |
| EUR senior unsecured (A bank) | FVA funding rate | 25–50 bps above ESTR |
| EUR Tier 2 / senior non-preferred | Capital-intensive funding | 60–120 bps |

### 3.2 Asymmetric Funding (s_f ≠ s_r)

Borrowing is typically more expensive than reinvesting: $s_f > s_r$. This asymmetry creates the **funding asymmetry premium** — the FBA credit for negative exposure is smaller than the FCA cost for positive exposure.

In the symmetric limit ($s_f = s_r = s$):

$$\text{FVA} = s \sum_i \Delta t_i \cdot \mathbb{E}^{\mathbb{Q}}[D(0,t_i)\,V(t_i)]$$

For an ATM trade at inception $\mathbb{E}[V(t_i)] \neq 0$ in general due to the convexity of the exposure distribution and OIS discounting effects — FVA is non-zero even for a par swap.

### 3.3 Cross-Currency Funding Basis

For EUR/USD cross-currency swaps, the USD funding spread must account for the EUR/USD cross-currency basis swap spread (typically −20 to −40 bps in 2026). A EUR bank funding USD collateral pays SOFR + basis + credit spread. This is modelled by adjusting $s_f$ for the non-domestic currency leg.


In [ ]:
# ── FVA sensitivity to funding spread and asymmetry ──────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: FVA vs funding spread (symmetric s_f = s_r)
ax = axes[0]
spread_grid = np.arange(0, 101, 5)
fva_symmetric = [
    compute_fva(profile['EE_disc'], nee_disc, exp_dates, s, s, 'none')['FVA']
    / SWAP_NOTIONAL * 10000
    for s in spread_grid
]
fca_only = [
    compute_fva(profile['EE_disc'], nee_disc, exp_dates, s, 0, 'none')['FCA']
    / SWAP_NOTIONAL * 10000
    for s in spread_grid
]

ax.plot(spread_grid, fca_only,    '-',  color='firebrick',  lw=1.8, label='FCA only (s_r=0)')
ax.plot(spread_grid, fva_symmetric,'--', color='darkorange', lw=1.8,
        label='FVA symmetric (s_f = s_r)')
ax.axvline(FUNDING_SPREAD_BPS, lw=0.8, color='black', ls=':', alpha=0.6,
           label=f'Reference s_f={FUNDING_SPREAD_BPS:.0f} bps')
ax.set_xlabel('Funding spread s_f (bps above OIS)')
ax.set_ylabel('FVA (bps of notional)')
ax.set_title(f'FVA vs funding spread — no CSA\n{direction} IRS, {SWAP_MATURITY:.0f}Y')
ax.legend(fontsize=7)

# Right: FVA vs asymmetry (s_f fixed, vary s_r)
ax = axes[1]
s_r_grid = np.arange(0, FUNDING_SPREAD_BPS + 1, 2)
fva_asym = [
    compute_fva(profile['EE_disc'], nee_disc, exp_dates,
                FUNDING_SPREAD_BPS, s_r, 'none')['FVA'] / SWAP_NOTIONAL * 10000
    for s_r in s_r_grid
]

ax.plot(s_r_grid, fva_asym, 'o-', color='purple', lw=1.8, ms=4)
ax.axvline(INVEST_SPREAD_BPS, color='black', lw=0.8, ls='--', alpha=0.6,
           label=f'Reference s_r={INVEST_SPREAD_BPS:.0f} bps')
ax.axvline(FUNDING_SPREAD_BPS, color='firebrick', lw=0.8, ls=':', alpha=0.6,
           label=f's_r = s_f (symmetric limit)')
ax.set_xlabel('Reinvestment spread s_r (bps)')
ax.set_ylabel('FVA (bps of notional)')
ax.set_title(f'FVA vs funding asymmetry\n(s_f={FUNDING_SPREAD_BPS:.0f} bps fixed)')
ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

print(f"At symmetric spread: FVA = {fva_symmetric[spread_grid==FUNDING_SPREAD_BPS][0]:.2f} bps")
print(f"At s_r=0 (full asymmetry): FCA = {fca_only[spread_grid==FUNDING_SPREAD_BPS][0]:.2f} bps")
print(f"FBA offset (s_r={INVEST_SPREAD_BPS:.0f} bps): {fca_only[spread_grid==FUNDING_SPREAD_BPS][0] - fva_symmetric[spread_grid==FUNDING_SPREAD_BPS][0]:.2f} bps reduction")


In [ ]:
# ── FVA sensitivity: swap direction × CSA type × HW vol ─────────────────────

print("=== FVA Sensitivity Matrix ===\n")

# 1. Swap direction
print("1. FVA by swap direction (no CSA):")
for is_pay, label in [(True,'Payer'), (False,'Receiver')]:
    def mtm_dir(paths, t):
        return irs_mtm(paths, t, payment_dates, year_fracs,
                       SWAP_COUPON, SWAP_NOTIONAL, is_pay,
                       KAPPA, SIGMA, ois, sim.dt, sim.n_steps)
    prof_dir  = sim.exposure_profile(mtm_dir, exp_dates)
    disc_d    = np.column_stack([sim.sdf(t) for t in exp_dates])
    nee_dir   = (disc_d * np.minimum(prof_dir['mtm'], 0)).mean(axis=0)
    res_dir   = compute_fva(prof_dir['EE_disc'], nee_dir, exp_dates,
                             FUNDING_SPREAD_BPS, INVEST_SPREAD_BPS, 'none')
    print(f"  {label}: FCA={res_dir['FCA']/SWAP_NOTIONAL*10000:.2f}  "
          f"FBA={res_dir['FBA']/SWAP_NOTIONAL*10000:.2f}  "
          f"FVA={res_dir['FVA']/SWAP_NOTIONAL*10000:.2f} bps")

# 2. HW vol — FVA grows with vol (wider exposure distribution)
print("\n2. FVA vs HW vol σ (no CSA, payer IRS):")
for sigma_v in [0.30, 0.50, 0.70, 1.00]:
    sim_v = MCSimulator(
        process  = HullWhiteProcess(curve=ois, kappa=KAPPA, sigma=sigma_v),
        x0=r0, T=10.0, n_steps=120, n_paths=5000, antithetic=True, seed=RNG_SEED,
    )
    def mtm_v(paths, t):
        return irs_mtm(paths, t, payment_dates, year_fracs,
                       SWAP_COUPON, SWAP_NOTIONAL, True,
                       KAPPA, sigma_v, ois, sim_v.dt, sim_v.n_steps)
    prof_v = sim_v.exposure_profile(mtm_v, exp_dates)
    disc_v = np.column_stack([sim_v.sdf(t) for t in exp_dates])
    nee_v  = (disc_v * np.minimum(prof_v['mtm'], 0)).mean(axis=0)
    res_v  = compute_fva(prof_v['EE_disc'], nee_v, exp_dates,
                          FUNDING_SPREAD_BPS, INVEST_SPREAD_BPS, 'none')
    print(f"  σ={sigma_v:.2f}%/√yr  EE_peak={prof_v['EE'].max()/SWAP_NOTIONAL*10000:.1f}bps  "
          f"FVA={res_v['FVA']/SWAP_NOTIONAL*10000:.2f} bps")

# 3. Notional scaling — FVA is linear in notional
print("\n3. Notional scaling (linear check):")
for notional_val in [500_000, 1_000_000, 5_000_000, 10_000_000]:
    def mtm_n(paths, t):
        return irs_mtm(paths, t, payment_dates, year_fracs,
                       SWAP_COUPON, notional_val, IS_PAYER,
                       KAPPA, SIGMA, ois, sim.dt, sim.n_steps)
    prof_n = sim.exposure_profile(mtm_n, exp_dates)
    disc_n = np.column_stack([sim.sdf(t) for t in exp_dates])
    nee_n  = (disc_n * np.minimum(prof_n['mtm'], 0)).mean(axis=0)
    res_n  = compute_fva(prof_n['EE_disc'], nee_n, exp_dates,
                          FUNDING_SPREAD_BPS, INVEST_SPREAD_BPS, 'none')
    print(f"  N={notional_val/1e6:.1f}M  FVA={res_n['FVA']:>10,.2f} EUR  "
          f"({res_n['FVA']/notional_val*10000:.2f} bps/notional)")


---
## Summary

| Step | Formula | Function |
|---|---|---|
| **FCA** | $s_f \sum_i \Delta t_i \cdot \text{EE}_{\text{disc}}(t_i)$ | `compute_fva(ee_disc, nee_disc, dates, s_f, s_r, csa_type)` |
| **FBA** | $s_r \sum_i \Delta t_i \cdot |\text{NEE}_{\text{disc}}(t_i)|$ | same |
| **FVA** | FCA − FBA | `result['FVA']` |
| **NEE_disc** | $\mathbb{E}[D(0,t)\min(V(t),0)]$ | `(disc_matrix * min(mtm,0)).mean(axis=0)` |
| **CSA two-way** | FCA = FBA = 0 → **FVA = 0** | `csa_type='two_way'` |
| **CSA one-way** | FBA = 0 → **FVA = FCA** | `csa_type='one_way_post'` |

**Key insights:**
1. **FVA is linear in the funding spread** — doubling $s_f$ doubles FCA
2. **CSA eliminates FVA entirely** — the value of a two-way CSA equals FVA(no CSA)
3. **FVA grows with rate vol** (σ) — wider exposure distribution → larger EE and |NEE|
4. **Payer vs Receiver swap**: for an ATM payer, EE > |NEE| (positive exposure dominates) → FVA > 0; receiver is approximately symmetric
5. **s_f ≠ s_r asymmetry** is the dominant driver of the FCA-FBA spread; $s_f = s_r$ is an approximation that underestimates FVA for most banks

**Next:** [QRE-55 — MVA](10_mva.ipynb) — Margin Valuation Adjustment: the cost of posting **initial margin** under SIMM (bilateral) or CCP (cleared). MVA uses the same exposure profile infrastructure but applies to the expected initial margin profile $\text{EIM}(t)$ rather than the trade MTM.
